#Dataset to Bengali

In [2]:
!pip install -q \
torch==2.6.0 \
torchvision==0.21.0 \
torchaudio==2.6.0 \
transformers==4.55.4 \
sentencepiece \
accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 816.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [3]:

"""
translate_nllb_pipeline.py

Offline English -> Bengali translation pipeline using
facebook/nllb-200-distilled-600M

Features
--------
- Merge train/valid/test
- Preserve labels
- GPU/CPU auto detection
- FP16 on CUDA
- Batch translation
- Checkpoint resume
- UTF-8 output
"""

import os
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TRAIN="train.csv"
VALID="valid.csv"
TEST="test.csv"

OUTDIR="output"
os.makedirs(OUTDIR,exist_ok=True)

MODEL_NAME="facebook/nllb-200-distilled-600M"

SRC_LANG="eng_Latn"
TGT_LANG="ben_Beng"

BATCH_SIZE=64 if torch.cuda.is_available() else 8
CHECKPOINT_EVERY=500

print("Loading datasets...")

train=pd.read_csv(TRAIN)
valid=pd.read_csv(VALID)
test=pd.read_csv(TEST)

df=pd.concat([train,valid,test],ignore_index=True)

df.to_csv(os.path.join(OUTDIR,"merged_english.csv"),index=False)

checkpoint=os.path.join(OUTDIR,"merged_bengali_checkpoint.csv")

translated=[]

if os.path.exists(checkpoint):
    cp=pd.read_csv(checkpoint)
    translated=cp["text"].tolist()
    print(f"Resuming from {len(translated)} rows")

device="cuda" if torch.cuda.is_available() else "cpu"
print("Device:",device)

print("Loading NLLB model...")
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

if device=="cuda":
    model=model.half()

model.to(device)
model.eval()

tokenizer.src_lang=SRC_LANG

start=len(translated)
texts=df["text"].astype(str).tolist()

def translate_batch(batch):
    inputs=tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        generated=model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_new_tokens=256,
            num_beams=4
        )

    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

count_since_checkpoint=0

for i in tqdm(range(start,len(texts),BATCH_SIZE)):
    batch=texts[i:i+BATCH_SIZE]
    out=translate_batch(batch)
    translated.extend(out)

    count_since_checkpoint+=len(out)

    if count_since_checkpoint>=CHECKPOINT_EVERY:
        tmp=df.iloc[:len(translated)].copy()
        tmp["text"]=translated
        tmp.to_csv(checkpoint,index=False,encoding="utf-8-sig")
        count_since_checkpoint=0
        print("Checkpoint saved:",len(translated))

df["text"]=translated

outfile=os.path.join(OUTDIR,"merged_bengali.csv")
df.to_csv(outfile,index=False,encoding="utf-8-sig")

print("Finished.")
print(outfile)


Loading datasets...
Device: cuda
Loading NLLB model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


 11%|█         | 8/76 [01:21<09:49,  8.67s/it]

... (15 outputs left out)


 95%|█████████▍| 72/76 [14:03<00:40, 10.01s/it]

Checkpoint saved: 4608



100%|██████████| 76/76 [14:28<00:00, 11.43s/it]

Finished.
output/merged_bengali.csv


# Dataset to hindi

In [1]:
!pip install -U torch transformers sentencepiece sacremoses pandas tqdm accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [1]:
!pip uninstall -y torch torchvision torchaudio transformers

['Found existing installation: torch 2.13.0',
 'Uninstalling torch-2.13.0:',
 '  Successfully uninstalled torch-2.13.0',
 'Found existing installation: torchvision 0.26.0+cu128',
 'Uninstalling torchvision-0.26.0+cu128:',
 '  Successfully uninstalled torchvision-0.26.0+cu128',
 'Found existing installation: torchaudio 2.11.0+cu128',
 'Uninstalling torchaudio-2.11.0+cu128:',
 '  Successfully uninstalled torchaudio-2.11.0+cu128',
 'Found existing installation: transformers 5.14.1',
 'Uninstalling transformers-5.14.1:',
 '  Successfully uninstalled transformers-5.14.1']

In [2]:
!pip install -q \
torch==2.6.0 \
torchvision==0.21.0 \
torchaudio==2.6.0 \
transformers==4.55.4 \
sentencepiece \
accelerate \
safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [4]:

"""
translate_hindi_nllb_robust.py
Robust offline English -> Hindi translation pipeline using
facebook/nllb-200-distilled-600M
"""

import os
import gc
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TRAIN="train.csv"
VALID="valid.csv"
TEST="test.csv"

MODEL="facebook/nllb-200-distilled-600M"
SRC="eng_Latn"
TGT="hin_Deva"

OUTDIR="output"
os.makedirs(OUTDIR,exist_ok=True)

MERGED=os.path.join(OUTDIR,"merged_english.csv")
CHECKPOINT=os.path.join(OUTDIR,"merged_hindi_checkpoint.csv")
FINAL=os.path.join(OUTDIR,"merged_hindi.csv")

MAX_LENGTH=256
BATCH_SIZE=32 if torch.cuda.is_available() else 8
CHECKPOINT_EVERY=500

device="cuda" if torch.cuda.is_available() else "cpu"
print("Device:",device)

print("Loading CSVs...")
df=pd.concat([
    pd.read_csv(TRAIN),
    pd.read_csv(VALID),
    pd.read_csv(TEST)
],ignore_index=True)

df.to_csv(MERGED,index=False)

texts=df["text"].fillna("").astype(str).tolist()

translations=[]

if os.path.exists(CHECKPOINT):
    cp=pd.read_csv(CHECKPOINT)
    translations=cp["text"].fillna("").astype(str).tolist()
    print("Resuming from",len(translations))

start=len(translations)

print("Loading tokenizer...")
tokenizer=AutoTokenizer.from_pretrained(MODEL)
tokenizer.src_lang=SRC

print("Loading model...")
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL)

if device=="cuda":
    model=model.half()

model.to(device)
model.eval()

# compatibility across transformers versions
if hasattr(tokenizer,"lang_code_to_id"):
    target_id=tokenizer.lang_code_to_id[TGT]
else:
    target_id=tokenizer.convert_tokens_to_ids(TGT)

def translate(batch):
    enc=tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    )
    enc={k:v.to(device) for k,v in enc.items()}

    with torch.no_grad():
        if device=="cuda":
            with torch.autocast("cuda"):
                out=model.generate(
                    **enc,
                    forced_bos_token_id=target_id,
                    max_new_tokens=MAX_LENGTH,
                    num_beams=1,
                    do_sample=False
                )
        else:
            out=model.generate(
                **enc,
                forced_bos_token_id=target_id,
                max_new_tokens=MAX_LENGTH,
                num_beams=1,
                do_sample=False
            )

    return tokenizer.batch_decode(out,skip_special_tokens=True)

def save():
    tmp=df.iloc[:len(translations)].copy()
    tmp["text"]=translations
    tmp.to_csv(CHECKPOINT,index=False,encoding="utf-8-sig")
    print(f"Checkpoint: {len(translations)}")

i=start
since=0
batch=BATCH_SIZE
pbar=tqdm(total=len(texts)-start)

while i<len(texts):
    try:
        result=translate(texts[i:i+batch])
        translations.extend(result)
        i+=len(result)
        since+=len(result)
        pbar.update(len(result))

        if since>=CHECKPOINT_EVERY:
            save()
            since=0

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        batch=max(1,batch//2)
        print("OOM. Reducing batch size to",batch)

pbar.close()
save()

df["text"]=translations
df.to_csv(FINAL,index=False,encoding="utf-8-sig")

print("Done!")
print(FINAL)


Device: cuda
Loading CSVs...
Loading tokenizer...
Loading model...


 11%|█         | 512/4856 [00:35<03:20, 21.62it/s]

Checkpoint: 512


 21%|██        | 1024/4856 [00:54<02:26, 26.08it/s]

Checkpoint: 1024


 32%|███▏      | 1536/4856 [01:18<02:07, 26.08it/s]

... (11 outputs left out)

 95%|█████████▍| 4608/4856 [03:44<00:08, 28.48it/s]

Checkpoint: 4608


100%|██████████| 4856/4856 [03:58<00:00, 20.39it/s]

Checkpoint: 4856
Done!
output/merged_hindi.csv
